In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv("input.csv")
df = df.dropna()
df

In [ ]:
categorical_cols = ["drug1", "drug2", "drug3", "interaction_type", "side_effect", "stage"]
for col in categorical_cols:
    print(f"\nUnique values in {col}: {df[col].nunique()}")
    print(df[col].value_counts())

In [ ]:
plt.figure(figsize=(10, 5))
sns.countplot(x="side_effect", data=df, order=df["side_effect"].value_counts().index)
plt.xticks(rotation=45)
plt.title("Distribution of Side Effects")
plt.show()

In [ ]:
plt.figure(figsize=(8, 4))
sns.countplot(x="stage", data=df)
plt.title("Distribution of Stage")
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
X = df[["drug1", "drug2", "drug3", "dosage1", "dosage2", "dosage3"]]
y = df["stage"]
import pickle
# Encode categorical variables
label_encoders = {}
for col in ["drug1", "drug2", "drug3"]:
    le = LabelEncoder()
    X[col] = le.fit_transform(X[col])
    
    label_encoders[col] = dict(zip(le.classes_, le.transform(le.classes_)))
with open("feature.pkl", "wb") as f:
        pickle.dump(label_encoders, f)




In [ ]:
# Train Random Forest model
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X, y)


In [ ]:
with open("randomforestmodelstage", "wb") as f:
        pickle.dump(rf_model, f)

In [ ]:
df = pd.read_csv("test.csv")
for col in ["drug1", "drug2", "drug3"]:
    df[col] = df[col].map(label_encoders[col])
#df = df.dropna()
df.fillna(0, inplace=True)
ytest = df["stage"]

Xtest = df[["drug1", "drug2", "drug3", "dosage1", "dosage2", "dosage3"]]

In [ ]:
Xtest

In [ ]:
y_pred = rf_model.predict(Xtest)

In [ ]:
accuracy = accuracy_score(ytest, y_pred)
print(f"Accuracy: {accuracy:.2f}")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
accuracy = accuracy_score(ytest, y_pred)
precision = precision_score(ytest, y_pred, average='weighted')
recall = recall_score(ytest, y_pred, average='weighted')
f1 = f1_score(ytest, y_pred, average='weighted')
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

In [ ]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, cm[i, j],
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, classification_report
from sklearn.metrics import confusion_matrix
import itertools
import numpy as np
# compute the confusion matrix
confusion_mtx = confusion_matrix(ytest, y_pred) 

plot_confusion_matrix(confusion_mtx, classes = range(4))